# 環境檢查

In [ ]:
# 🎯 確認這個 kernel 就是課程環境，讀這台機器對應的環境檔，把缺的套件裝起來，再印出每一個套件的版本。

# ---- 1. 內建函式庫、這一本共用的幾個小工具，以及關掉套件的雜訊訊息 ----
#         CHECKS 收每一項檢查的結果，最後一格再把它排成總表。
from pathlib import Path
from importlib import metadata
from IPython.display import display
import importlib
import logging
import os
import platform
import re
import subprocess
import sys
import warnings

CHECKS = []


def record(item, status, detail, fix=""):
    """把一項檢查的結果寫進 CHECKS。狀態只有三種: 通過 / 失敗 / 注意。

    同一個檢查項目重跑的時候是換掉舊的那一列，不是再加一列。第四格會請學員裝完 Docker
    再回到那一格重跑，沒有這一段的話總表會多出一份重複的列，「通過 N 項」也跟著灌水。
    """
    row = {"檢查項目": item, "狀態": status,
           "實際結果": str(detail), "沒過的話怎麼辦": fix}
    for i, existing in enumerate(CHECKS):
        if existing["檢查項目"] == item:
            CHECKS[i] = row
            return status, detail
    CHECKS.append(row)
    return status, detail


def safe(item, func, fix=""):
    """跑一項檢查，並且保證它出錯的時候這一格不會中斷，只會在總表多一列失敗。"""
    try:
        status, detail = func()
    except Exception as exc:
        status, detail = "失敗", f"{type(exc).__name__}: {exc}"
    return record(item, status, detail, fix)


def show_table(rows, columns):
    """把幾列資料排成一張表。pandas 有可能是這一格才剛裝好的，所以在函式裡面才 import。"""
    try:
        import pandas as pd
        pd.set_option("display.max_colwidth", 200)   # 不設的話「怎麼辦」那一欄會被截掉
        pd.set_option("display.width", 200)
        display(pd.DataFrame(rows, columns=columns))
    except Exception:
        print(" | ".join(columns))
        for row in rows:
            print(" | ".join(str(x) for x in row))


# Prophet 與它的後端 cmdstanpy 在 import 時會印出好幾行訊息與提示，先關掉，表格才看得清楚
# (兩本 lab 的第一格做的是同一件事)。
warnings.filterwarnings("ignore")
for _log_name in ("cmdstanpy", "prophet", "stan"):
    _lg = logging.getLogger(_log_name)
    _lg.setLevel(logging.CRITICAL)
    _lg.handlers.clear()
    _lg.addHandler(logging.NullHandler())
    _lg.propagate = False

# ---- 2. 這台機器要用哪一份環境檔，week6_implementation 資料夾在哪裡 ----
#         這門課只支援 macOS 與 Windows，兩份環境檔差一個 pywin32。
ENV_NAME = "aiops-anomaly-zero-to-hero"
OS_LABEL = {"Darwin": "macOS", "Windows": "Windows"}.get(platform.system(), platform.system())
ENV_YML = {"macOS": "environment.macos.yml", "Windows": "environment.windows.yml"}.get(OS_LABEL)

# 從現在的資料夾往上走，找到帶著 environments/ 的那一層，那一層就是 week6_implementation。
# 兩本 lab 找 data/ 用的是同一套走法，所以這裡算出來的位置跟它們一定一致。
WEEK6_DIR = Path.cwd()
while WEEK6_DIR != WEEK6_DIR.parent and not (WEEK6_DIR / "environments").is_dir():
    WEEK6_DIR = WEEK6_DIR.parent
if not (WEEK6_DIR / "environments").is_dir():
    WEEK6_DIR = Path.cwd()

if ENV_YML is None:
    FIX_CONDA = f"這門課只支援 macOS 與 Windows，這台機器是 {platform.system()}，請換一台上課"
    record("作業系統", "失敗", f"{platform.system()} 沒有對應的環境檔", FIX_CONDA)
else:
    # Windows 的路徑分隔符號是反斜線，這一行直接寫成學生可以照打的樣子。
    _yml_path = f"environments\\{ENV_YML}" if OS_LABEL == "Windows" else f"environments/{ENV_YML}"
    FIX_CONDA = (f"開 Anaconda Prompt (macOS 開終端機)，cd 到 week6_implementation 資料夾，"
                 f"跑 conda activate {ENV_NAME}，再跑 conda env update -f {_yml_path} "
                 f"(不要加 --prune，那會把別的 lab 需要的套件刪掉)")

record("Python 版本", "通過" if sys.version_info[:2] >= (3, 10) else "失敗",
       f"Python {platform.python_version()}", FIX_CONDA)

# ---- 3. 這個 kernel 是不是課程環境 ----
#         這一項排在最前面，因為它決定其他每一項的意思。這一本量的是「現在這個 kernel」;
#         兩本 lab 如果選到另一個 kernel，這一整本的結果就跟它們無關，全部通過也沒有用。
#         conda 環境的資料夾名字就是環境名字，所以直譯器路徑的上一層可以當第二個依據
#         (CONDA_DEFAULT_ENV 不一定會傳進 Jupyter 的 kernel) 。
FIX_KERNEL = (f"在 JupyterLab 右上角按 kernel 名字 (或 Kernel > Change Kernel) ，"
              f"選 {ENV_NAME}。清單裡沒有的話，先 conda activate {ENV_NAME}，"
              f"再跑 python -m ipykernel install --user --name {ENV_NAME}，重開 JupyterLab")
#         venv 型的環境資料夾常常就叫 .venv，環境名字在上一層，所以兩層都看。
_env_by_var = os.environ.get("CONDA_DEFAULT_ENV")
_prefix = Path(sys.prefix)
_env_by_path = _prefix.parent.name if _prefix.name == ".venv" else _prefix.name
IN_COURSE_ENV = ENV_NAME in (_env_by_var, _env_by_path)
record("kernel 是課程環境",
       "通過" if IN_COURSE_ENV else "失敗",
       f"這個 kernel 是 {_env_by_var or _env_by_path}" if IN_COURSE_ENV
       else f"這個 kernel 是 {_env_by_var or _env_by_path}，不是 {ENV_NAME}。"
            f"兩本 lab 請用跟這一本同一個 kernel，不然這一本檢查的不是它們會用的環境",
       FIX_KERNEL)

# ---- 4. Windows 專屬: 資料夾路徑有中文或空白的時候，prophet 的編譯步驟會失敗 ----
#         這一項要在裝套件之前講，不然學員是等 prophet 裝到一半爆掉才知道。
if OS_LABEL == "Windows":
    _bad_path = (not str(WEEK6_DIR).isascii()) or (" " in str(WEEK6_DIR))
    record("資料夾路徑 (Windows) ",
           "注意" if _bad_path else "通過",
           f"{WEEK6_DIR}" + ("  路徑有中文或空白" if _bad_path else "  只有英數字"),
           "把 week6_implementation 整個資料夾搬到 C:\\week6 再重跑一次。"
           "Windows 使用者名稱含中文或空白時，prophet 背後的編譯步驟會失敗")

show_table([
    ("作業系統", platform.platform()),
    ("Python 版本", platform.python_version()),
    ("Python 直譯器", sys.executable),
    ("這個 kernel 的環境", f"{_env_by_var or _env_by_path}" + ("" if IN_COURSE_ENV else f"  (課程環境是 {ENV_NAME}) ")),
    ("week6_implementation 資料夾", str(WEEK6_DIR)),
    ("這台機器該用的環境檔", f"environments/{ENV_YML}" if ENV_YML else "沒有，只支援 macOS 與 Windows"),
], ["項目", "這台機器"])

# ---- 5. 從環境檔讀出套件清單，逐一 import 看缺哪些 ----
#         安裝時打的名字跟 import 時打的名字不一定一樣，CONDA_TO_IMPORT 就是在對照這件事。
#         NOT_A_PACKAGE 裡的四個不進 import 檢查:
#           python、pip  不是拿來 import 的東西
#           jupyterlab   是「開這本 notebook 的那個程式」，不是這個 kernel 裡的東西。在 kernel 裡
#                        import 得到它，跟畫面上那個 JupyterLab 是哪一版沒有關係，所以改用第三格
#                        的 mermaid 流程圖用眼睛驗
#           pywin32      Week 6 沒有任何一支程式 import 它，而且它靠一個 .pth 檔把 win32 加進
#                        搜尋路徑，那個檔只有直譯器啟動時才會讀，pip 裝完當下 import 一定失敗，
#                        留著只會製造一列假的失敗
CONDA_TO_IMPORT = {"scikit-learn": "sklearn", "ipython": "IPython"}
NOT_A_PACKAGE = {"python", "pip", "jupyterlab", "pywin32"}


def parse_env_yml(path):
    """拆成 (conda 名字, 帶版本條件的完整字串)。刻意不用 yaml 套件: 讀環境檔的時候還不能假設有第三方套件。"""
    specs, inside = [], False
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.split("#", 1)[0].rstrip()                # 砍掉註解
        if line.startswith("dependencies:"):
            inside = True
            continue
        if inside and line and not line.startswith(" "):    # 回到最外層就代表這一段結束
            break
        if inside and line.strip().startswith("- "):
            spec = line.strip()[2:].strip()
            # 套件名字是第一個版本符號之前的那一段，例如 scikit-learn>=1.4,<2 的 scikit-learn。
            name = re.split(r"[=<>!~]", spec, maxsplit=1)[0].strip()
            if name and name not in NOT_A_PACKAGE:
                specs.append((name, spec))
    return specs


_env_path = (WEEK6_DIR / "environments" / ENV_YML) if ENV_YML else None
if _env_path is not None and _env_path.is_file():
    SPECS = parse_env_yml(_env_path)
else:
    SPECS = []
    record("環境檔在不在", "失敗",
           f"找不到 {_env_path}" if _env_path else f"{platform.system()} 沒有對應的環境檔",
           "確認 JupyterLab 是在 week6_implementation 資料夾裡打開的，而且 environments/ 這個資料夾還在")

# 環境檔讀不到的時候，下面十幾項套件檢查一項都不會跑。沒有這一列的話總表看起來只少了一項，
# 學員會以為只是小事。
if not SPECS:
    record("套件檢查有沒有執行", "失敗", "環境檔讀不到，這一格的套件檢查一項都沒有跑",
           "先處理上面「環境檔在不在」那一列，再重跑這一格")


def missing_now():
    """回傳現在 import 不進來的那幾個 (conda 名字, 完整字串)。"""
    out = []
    for name, spec in SPECS:
        try:
            importlib.import_module(CONDA_TO_IMPORT.get(name, name))
        except Exception:
            out.append((name, spec))
    return out


MISSING = missing_now()

# ---- 6. 缺的直接裝起來 ----
#         用 subprocess 而不是 %pip: %pip 裝失敗的時候不會丟出例外，回傳碼也拿不到，
#         所以裝壞了這一本會當成沒事。sys.executable 就是這個 kernel 的 python，
#         所以裝到的一定是這個 kernel 在用的環境，Windows 也一樣。
def pip_install(spec):
    """裝一個套件，回傳 (成功與否, 一行訊息) 。"""
    proc = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", spec],
                          capture_output=True, text=True)
    if proc.returncode == 0:
        return True, "裝好了"
    tail = [ln for ln in (proc.stderr or proc.stdout).splitlines() if ln.strip()]
    return False, (tail[-1][:180] if tail else f"pip 回傳碼 {proc.returncode}")


def internet_ok():
    """連得到 PyPI 嗎。連不到的話下面每一個 pip install 都會失敗，先講比較省時間。"""
    from urllib.request import urlopen
    try:
        with urlopen("https://pypi.org/simple/", timeout=8):
            return True
    except Exception:
        return False


INSTALL_FAILED = []
if MISSING:
    print("這台機器缺這幾個，現在裝: " + "、".join(name for name, _ in MISSING))
    if not internet_ok():
        record("連得到 PyPI", "失敗", "連不到 pypi.org，接下來的安裝都會失敗",
               "確認這台機器連得上網路 (教室 Wi-Fi 要先在瀏覽器按同意) ，再重跑這一格")
        print("連不到 pypi.org: 先把網路接通，再重跑這一格。")
    else:
        record("連得到 PyPI", "通過", "連得到 pypi.org", "")
        for pkg_name, pkg_spec in MISSING:
            ok, note = pip_install(pkg_spec)
            print(f"  {pkg_name}: {note}")
            if not ok:
                INSTALL_FAILED.append((pkg_name, note))
else:
    print("全部都在，不用裝。")

for pkg_name, note in INSTALL_FAILED:
    record(f"安裝 {pkg_name}", "失敗", note, FIX_CONDA)

importlib.invalidate_caches()        # 剛裝好的套件要重掃一次，這個 kernel 才 import 得到
STILL_MISSING = missing_now() if MISSING else []

if STILL_MISSING:
    print("\n裝不起來的: " + "、".join(name for name, _ in STILL_MISSING))
    if OS_LABEL == "Windows" and any(name == "prophet" for name, _ in STILL_MISSING):
        print("prophet 在 Windows 上用 pip 裝要先編譯，裝不起來多半卡在這一步。")
    print("請改用 conda 裝: " + FIX_CONDA)
    print("裝完關掉 JupyterLab 重開，再跑一次這一本。")
elif MISSING:
    # 這裡刻意不叫學員重開 kernel: 上面已經 import 成功了，就證明這個 kernel 現在拿得到,
    # 重開一次要多花五分鐘，而且回來看到的表跟現在這一張一模一樣。
    print("\n裝好了，而且這個 kernel 現在都 import 得到，不用重開 kernel。")

# ---- 7. 印出每一個套件的版本，並且比對環境檔上的版本條件 ----
#         只 import 得進來還不夠: 版本太舊一樣 import 得到，但兩本 lab 會在中途壞掉。
#         packaging 是 matplotlib 的相依套件，所以裝得到 matplotlib 就一定有它;
#         真的沒有的話就只印版本不比對，不讓這一格中斷。
try:
    from packaging.specifiers import SpecifierSet
    from packaging.version import Version
except Exception:
    SpecifierSet = None

rows = []
for name, spec in SPECS:
    import_name = CONDA_TO_IMPORT.get(name, name)
    condition = spec[len(name):].strip()          # numpy>=1.26,<3 的 >=1.26,<3

    def check_import(import_name=import_name, name=name, condition=condition):
        module = importlib.import_module(import_name)
        # 大多數套件有 __version__；沒有的 (例如 prometheus_client) 就去查已安裝套件的中繼資料。
        version = str(getattr(module, "__version__", None) or metadata.version(name))
        if condition and SpecifierSet is not None:
            if not SpecifierSet(condition).contains(Version(version), prereleases=True):
                return "失敗", f"{version}，但環境檔要求 {condition}"
        return "通過", version

    status, detail = safe(f"套件 {name}", check_import, FIX_CONDA)
    rows.append((name, import_name, condition or "沒有限制", status, detail))

show_table(rows, ["環境檔上的名字", "import 時打的名字", "版本條件", "狀態", "版本或錯誤訊息"])

# ---- 8. prophet 真的 fit 得起來嗎 ----
#         import 成功不代表 fit 得起來: prophet 背後是 cmdstan，conda 跟 pip 混裝的時候
#         import 沒事、fit 的時候才爆。Lab 06 從頭到尾都在 fit，所以這裡先用 40 個點試一次,
#         大約零點幾秒。
def prophet_fits():
    import pandas as pd
    from prophet import Prophet
    frame = pd.DataFrame({"ds": pd.date_range("2026-01-01", periods=40, freq="D"),
                          "y": [10.0 + (i % 7) for i in range(40)]})
    Prophet(weekly_seasonality=True, yearly_seasonality=False,
            daily_seasonality=False).fit(frame)
    return "通過", "40 個點 fit 得起來"


safe("prophet 跑得動", prophet_fits,
     "prophet 是這門課最容易裝壞的套件。" + FIX_CONDA)

# ---- 9. 兩本 lab 第一格會用的 matplotlib 樣式 ----
def style_available():
    import matplotlib.pyplot as plt
    name = "seaborn-v0_8-whitegrid"
    if name in plt.style.available:
        return "通過", name
    return "失敗", f"這一版 matplotlib 沒有 {name}"


_ = safe("matplotlib 樣式", style_available,
         "兩本 lab 的第一格會用這個樣式。" + FIX_CONDA)


In [ ]:
# 🎯 找一個中文字型設給 matplotlib，畫一張中文圖，請用眼睛確認圖上是中文字不是空心方框。

# ---- 1. 畫圖要用的東西，以及這一本顯示圖的方式 ----
import base64
import io
import matplotlib.pyplot as plt
import matplotlib.font_manager as _fm
from IPython.display import HTML


def show_fig(fig, width="80%"):
    """把圖存成 PNG，再以 80% 寬度置中顯示，然後關掉它。

    直接 display(fig) 會靠左，寬度也只由 figsize 決定; 轉成 <img> 才控制得了寬度與置中。
    """
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=fig.dpi, bbox_inches="tight")
    plt.close(fig)
    src = "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode("ascii")
    display(HTML(f'<div style="text-align: center">'
                 f'<img src="{src}" style="width: {width}"></div>'))


# ---- 2. 找一個裝得到的中文字型設給 matplotlib ----
#         作業系統內建的中文字型名稱不一樣，由左到右試，用第一個裝得到的
#         (這份候選清單跟兩本 lab 第一格用的完全相同)。
CJK_CANDIDATES = ["PingFang TC", "PingFang HK", "Heiti TC", "Arial Unicode MS",   # macOS
                  "Microsoft JhengHei", "Microsoft YaHei",                        # Windows
                  "Noto Sans CJK TC", "Noto Sans TC", "Source Han Sans TW",
                  "WenQuanYi Zen Hei", "Droid Sans Fallback"]
FONT_FIX = {
    "macOS": "系統內建 PingFang TC，正常情況不會走到這裡。真的找不到就把一個 .otf 或 .ttf "
             "中文字型檔放進 week6_implementation/fonts/，再重跑這一格",
    "Windows": "系統內建微軟正黑體 Microsoft JhengHei。英文版 Windows 可能沒有裝，"
               "到「設定 > 時間與語言 > 語言」加入「中文 (繁體，台灣) 」語言套件，"
               "或把一個 .otf 或 .ttf 中文字型檔放進 week6_implementation\\fonts\\，再重跑這一格",
}.get(OS_LABEL, "把一個中文字型檔放進 week6_implementation/fonts/，再重跑這一格")


def pick_cjk_font():
    """回傳選到的中文字型名字，選不到就回傳 None。

    照三個順序試，前一個不行才換下一個:
      1. matplotlib 現在已經認得的字型
      2. 重建一次字型快取再找。字型是裝好了但 matplotlib 的快取還是舊的，就會卡在這裡,
         而這是這一項最常見的失敗方式，重建完通常就找得到
      3. week6_implementation/fonts/ 底下自備的字型檔。前兩步都沒有的時候 (例如英文版
         Windows 沒裝中文語言套件) ，把字型檔放進那個資料夾就能救回來，不必改系統設定
    """
    installed = {f.name for f in _fm.fontManager.ttflist}
    found = next((name for name in CJK_CANDIDATES if name in installed), None)
    if found:
        return found, "系統字型"

    # 第二步: 重建快取。try_read_cache=False 會重新掃一次系統字型資料夾。
    try:
        _fm._load_fontmanager(try_read_cache=False)
        installed = {f.name for f in _fm.fontManager.ttflist}
        found = next((name for name in CJK_CANDIDATES if name in installed), None)
        if found:
            return found, "重建字型快取之後找到的"
    except Exception:
        pass

    # 第三步: 資料夾裡自備的字型檔。
    font_dir = WEEK6_DIR / "fonts"
    if font_dir.is_dir():
        for path in sorted(font_dir.glob("*.[ot]tf")):
            try:
                _fm.fontManager.addfont(str(path))
                return _fm.FontProperties(fname=str(path)).get_name(), f"fonts/{path.name}"
            except Exception:
                continue
    return None, "找不到"


CJK_FONT, CJK_SOURCE = pick_cjk_font()
if CJK_FONT:
    plt.rcParams["font.sans-serif"] = [CJK_FONT] + plt.rcParams["font.sans-serif"]
    # 中文字型多半沒有獨立的粗體檔，matplotlib 會為每一次粗體字送出一行提示，關掉。
    logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
plt.rcParams["axes.unicode_minus"] = False     # 用 ASCII 的減號，中文字型才畫得出負號

# ---- 3. 畫一張中文圖，這一項要你自己用眼睛看 ----
#         版本號跟檔案在不在可以用程式判斷，字有沒有真的被畫出來不行。
fig, ax = plt.subplots(figsize=(6.6, 2.6))
ax.plot([0, 1, 2, 3, 4], [1.0, 3.0, 2.0, 4.0, 3.5], color="tab:red", marker="o", label="流量")
ax.set_title("中文字型測試: 這一行要看得到字，不能是空心方框")
ax.set_xlabel("時間 (小時) ")
ax.set_ylabel("流量 (bytes) ")
ax.legend(loc="upper left")
# transform=ax.transAxes 的意思是這一行字用「圖框的比例」定位，0.60 與 0.08 就是
# 從左邊算 60%、從下面算 8% 的地方，不會壓到上面那條線。
ax.text(0.60, 0.08, "負號要畫得出來: -12.5", transform=ax.transAxes)
fig.tight_layout()
show_fig(fig)

record("matplotlib 畫得出中文",
       "通過" if CJK_FONT else "失敗",
       f"用的是 {CJK_FONT} ({CJK_SOURCE}) ，請確認上面那張圖是中文字" if CJK_FONT
       else "候選清單裡的中文字型一個都沒裝到，上面那張圖的中文會是空心方框",
       FONT_FIX)
show_table([(CHECKS[-1]["檢查項目"], CHECKS[-1]["狀態"], CHECKS[-1]["實際結果"])],
           ["項目", "狀態", "實際結果"])


## JupyterLab 版本檢查: 下面應該是一張流程圖

這一項也要你自己用眼睛看。**下面應該是一張由三個方塊連起來的流程圖。如果你看到的是一段
文字而不是圖，代表你的 JupyterLab 太舊**,兩本 lab 裡的流程圖 (Lab 06 有 7 張、Lab 07 有 22 張)
全部都會變成這個樣子。

這一項沒辦法用程式判斷: 畫流程圖的是「開這本 notebook 的那個 JupyterLab」，不是這個 kernel,
在 kernel 裡 import 到的 jupyterlab 版本跟畫面上這一個是不是同一個沒有關係。

沒看到圖的話: 關掉 JupyterLab，開 Anaconda Prompt (macOS 開終端機) ，
`conda activate aiops-anomaly-zero-to-hero`，跑 `conda install -c conda-forge "jupyterlab>=4.3"`,
再 `jupyter lab` 重開。

```mermaid
flowchart LR
    A[看得到這三個方塊] --> B[JupyterLab 版本夠新]
    B --> C[兩本 lab 的流程圖都畫得出來]
```


In [ ]:
# 🎯 確認兩本 lab 會讀到的每一份檔案都在: 三份 CSV、錄好的 LLM 回應、圖檔、Grafana 那一包。

# ---- 1. 三份資料檔，以及這一版教材裡它們應該長什麼樣子 ----
#         列數欄數對不上，通常是用 Excel 開過再存檔，或是下載中斷留下半個檔案。
import json
import pandas as pd

DATA_DIR = WEEK6_DIR / "data" / "synthetic"
DATA_FILES = [
    ("synthetic_rrd_metrics_week6.csv", "指標檔: 五個 port、一整個月、每 5 分鐘一列", 43200, 21),
    ("synthetic_event_catalog_week6.csv", "事件目錄: 每一次注入的事件在哪個 port、從何時到何時", 17, 7),
    ("synthetic_scheduled_calendar_week6.csv", "排程行事曆: 開盤微爆、收盤集合競價、日終對帳", 107, 7),
]
_cd = "cd 進 week6_implementation 再打 jupyter lab"
FIX_DATA = (f"先確認 JupyterLab 是在 week6_implementation 資料夾裡打開的 "
            f"(開 Anaconda Prompt，macOS 開終端機，{_cd}) ，"
            f"再確認 data/synthetic/ 底下三份 CSV 都在，缺的話重新下載一次整包教材")

# ---- 2. 逐一讀進來，跟預期的列數欄數比對 ----
def read_one(path, rows_expected, cols_expected):
    if not path.is_file():
        return "失敗", f"找不到 {path}"
    table = pd.read_csv(path)
    detail = f"{len(table):,} 列 {table.shape[1]} 欄"
    if "timestamp" in table.columns:      # 只有指標檔有時間欄，順便報時間範圍與 port 個數
        stamps = pd.to_datetime(table["timestamp"])
        detail += (f"，{stamps.min():%Y-%m-%d %H:%M} 到 {stamps.max():%Y-%m-%d %H:%M}"
                   f"，{table['port_id'].nunique()} 個 port")
    if (len(table), table.shape[1]) != (rows_expected, cols_expected):
        return "失敗", f"{detail}，但這一版教材應該是 {rows_expected:,} 列 {cols_expected} 欄"
    return "通過", detail


rows = []
for filename, purpose, n_rows, n_cols in DATA_FILES:
    # 預設參數 p=... 的用意是把這一圈的值固定住，不然函式要到 safe() 裡面才執行，
    # 會拿到迴圈最後一圈的值。
    status, detail = safe(f"資料 {filename}",
                          lambda p=DATA_DIR / filename, r=n_rows, c=n_cols: read_one(p, r, c),
                          FIX_DATA)
    rows.append((filename, status, detail, purpose))

show_table(rows, ["檔名", "狀態", "實際結果", "這一份是什麼"])

# ---- 3. 資料檔以外，兩本 lab 與 Grafana 還會讀到的東西 ----
#         這幾份少了一樣會讓課停下來，所以在這裡一起確認，不要等到跑到那一格才發現。
#         llm_diagnoses.json 特別要看內容: Lab 07 沒有 API 金鑰的時候整章靠它重播，
#         檔案在但少了 diagnose 或 rank_causes，那一格會直接結束。
def check_llm_recording():
    path = WEEK6_DIR / "llm_diagnoses.json"
    if not path.is_file():
        return "失敗", f"找不到 {path.name}"
    data = json.loads(path.read_text(encoding="utf-8"))
    missing = [k for k in ("diagnose", "rank_causes", "meta") if k not in data]
    if missing:
        return "失敗", f"{path.name} 少了 {'、'.join(missing)}"
    events = sorted(set(data["diagnose"]) & set(data["rank_causes"]))
    if "L" not in events:
        return "失敗", f"{path.name} 裡沒有事故 L 的回應 (現在有 {events}) "
    return "通過", f"錄了事故 {'、'.join(events)}，模型 {data['meta'].get('model', '未知')}"


safe("Lab 07 錄好的 LLM 回應", check_llm_recording,
     "llm_diagnoses.json 要跟兩本 notebook 放在同一層。缺的話重新下載一次整包教材")


def check_supporting_files():
    """兩支 .py 與 Grafana 那一包。少了它們前面兩本 lab 照跑，但最後一步開不起來。"""
    needed = ["4_grafana.py", "results_exporter.py",
              "infra/stack/compose.yaml", "infra/stack/prometheus/prometheus.yml",
              "infra/stack/replayer/Dockerfile",
              "infra/stack/grafana/dashboards/lab06-forecast-case.json",
              "infra/stack/grafana/dashboards/lab07-rca-case.json"]
    gone = [rel for rel in needed if not (WEEK6_DIR / rel).is_file()]
    if gone:
        return "失敗", "少了 " + "、".join(gone)
    return "通過", f"{len(needed)} 份都在"


safe("Grafana 那一包的檔案", check_supporting_files,
     "重新下載一次整包教材，infra/ 這個資料夾要完整")


def check_images():
    """兩本 lab 內嵌的講義圖與螢幕截圖，一張一張對。

    刻意不寫「應該有幾張」: 那個數字每次增刪圖都要跟著改，忘了改就變成一個永遠不會響的檢查。
    改成直接從兩本 notebook 的原始碼把 <img src="..."> 的路徑撈出來，再看那個檔案在不在,
    要求就永遠跟教材本身一致。少一張只會變成破圖，不會中斷，所以標注意。
    """
    wanted = set()
    for name in ("2_lab06_forecasting.ipynb", "3_lab07_root_cause_analysis.ipynb"):
        path = WEEK6_DIR / name
        if not path.is_file():
            return "失敗", f"找不到 {name}"
        wanted |= set(re.findall(r'(?:slides|screenshots)/[A-Za-z0-9_.-]+\.png',
                                 path.read_text(encoding="utf-8")))
    gone = sorted(rel for rel in wanted if not (WEEK6_DIR / rel).is_file())
    if gone:
        return "注意", f"兩本 lab 引用 {len(wanted)} 張，少了 {len(gone)} 張: " + "、".join(gone)
    return "通過", f"兩本 lab 引用的 {len(wanted)} 張圖都在"


safe("講義圖與截圖", check_images,
     "重新下載一次整包教材。這一項沒過只會讓 notebook 裡的圖變成破圖，分析照跑")

# ---- 4. 兩本 lab 要寫結果的資料夾 ----
#         直接建起來，不用麻煩學員。壓縮軟體有時候會把空資料夾吃掉。
(WEEK6_DIR / "outputs" / "workshop").mkdir(parents=True, exist_ok=True)
record("結果資料夾", "通過", "outputs/workshop/ 已備妥", "")

show_table([(c["檢查項目"], c["狀態"], c["實際結果"]) for c in CHECKS
            if c["檢查項目"].startswith(("Lab 07 錄", "Grafana 那", "講義圖", "結果資料夾"))],
           ["項目", "狀態", "實際結果"])


In [ ]:
# 🎯 檢查 docker 指令、Docker 背景服務、四個 host port，並且先把 Grafana 要用的映像檔抓下來。

# ============ ⚙️ 可調參數 ============
# DOCKER_TIMEOUT: 等 docker info 回應的秒數。Docker Desktop 正在啟動的期間這個指令要等很久，
#              設太短會把「還在啟動」誤判成「沒有在跑」。
# PREPULL:     Docker 可以用的時候，要不要在背景先把三個映像檔抓下來 (約 600 MB 到 1 GB) 。
#              不先抓的話，這些下載會全部擠在下課前跑 4_grafana.py 的那幾分鐘，
#              而且是全班同時抓。設 False 可以關掉。
DOCKER_TIMEOUT = 20
PREPULL = True
# ====================================

# 四個 host port 不寫在這裡。4_grafana.py 已經有一份 DEFAULT_PORTS，而且它會讀
# infra/stack/.env; 這一格如果自己寫死一份，學員照建議改了 .env 之後，這裡檢查的
# 還是舊的那四個，等於沒檢查。所以直接把那支程式讀進來問它。
import errno
import importlib.util
import shutil
import socket
import subprocess

# ---- 1. 先確認找得到 docker 指令 ----
#         JupyterLab 是在它啟動的那一刻決定 PATH 的，所以「裝完 Docker 再回來重跑這一格」
#         這件事本來會失敗: 新裝的 docker 不在這個 kernel 的 PATH 裡。這裡把 Docker Desktop
#         實際會裝到的幾個位置補進 PATH，學員就真的不用重開 JupyterLab。
_EXTRA_PATHS = {
    "macOS": ["/usr/local/bin", "/opt/homebrew/bin", "/Applications/Docker.app/Contents/Resources/bin"],
    "Windows": [os.path.expandvars(r"%ProgramFiles%\Docker\Docker\resources\bin"),
                os.path.expandvars(r"%ProgramFiles%\Docker\Docker\resources")],
}.get(OS_LABEL, [])
for _extra in _EXTRA_PATHS:
    if Path(_extra).is_dir() and _extra not in os.environ.get("PATH", ""):
        os.environ["PATH"] = os.environ.get("PATH", "") + os.pathsep + _extra

FIX_DOCKER = ("到 https://www.docker.com/products/docker-desktop/ 裝 Docker Desktop 並打開它，"
              "等圖示不再跑動，再回到這一格重跑")

DOCKER_PATH = shutil.which("docker")
DAEMON_OK = False
COMPOSE_OK = False
if DOCKER_PATH is None:
    docker_detail = "這台機器上沒有 docker 指令"
else:
    # docker info 沒有 timeout 的話，Docker Desktop 正在啟動的期間這一格會一直卡著;
    # 有 timeout 但沒有接住 TimeoutExpired 的話，這一格會直接中斷，後面的 port 檢查
    # 一項都不會跑，而總表看起來只是少了幾列，沒有人會發現。
    try:
        probe = subprocess.run(["docker", "info"], capture_output=True, text=True,
                               timeout=DOCKER_TIMEOUT)
        DAEMON_OK = probe.returncode == 0
        docker_detail = (f"docker 指令在 {DOCKER_PATH}，背景服務有回應" if DAEMON_OK
                         else "docker 裝好了，但背景服務沒有在跑，要先打開 Docker Desktop")
    except subprocess.TimeoutExpired:
        docker_detail = (f"docker info 在 {DOCKER_TIMEOUT} 秒內沒有回應，"
                         "Docker Desktop 可能還在啟動")
    except Exception as exc:
        docker_detail = f"{type(exc).__name__}: {exc}"

rows = [("docker", "通過" if DAEMON_OK else "注意", docker_detail)]
record("Docker 可以用", "通過" if DAEMON_OK else "注意", docker_detail, FIX_DOCKER)

# ---- 2. docker compose 這個外掛 ----
#         4_grafana.py 每一個動作都是 docker compose，但 docker info 過了不代表有這個外掛。
#         它不需要背景服務，所以就算 Docker 還沒打開也問得到答案。
if DOCKER_PATH is not None:
    try:
        probe = subprocess.run(["docker", "compose", "version"], capture_output=True,
                               text=True, timeout=DOCKER_TIMEOUT)
        COMPOSE_OK = probe.returncode == 0
        compose_detail = (probe.stdout.strip().splitlines()[0] if COMPOSE_OK
                          else "有 docker，但沒有 docker compose 這個外掛")
    except Exception as exc:
        compose_detail = f"{type(exc).__name__}: {exc}"
else:
    compose_detail = "沒有 docker 指令，這一項不用問"
record("docker compose", "通過" if COMPOSE_OK else "注意", compose_detail,
       "更新 Docker Desktop 到新版，compose 是它內附的外掛")
rows.append(("docker compose", "通過" if COMPOSE_OK else "注意", compose_detail))

# ---- 3. 四個 host port，問 4_grafana.py 要現在生效的那一組 ----
def load_grafana_helper():
    """把 4_grafana.py 讀進來當模組用。檔名是數字開頭，一般的 import 打不出來，要用這一招。
    那支程式的 main() 有 __name__ == "__main__" 擋著，所以讀進來不會做任何事。"""
    path = WEEK6_DIR / "4_grafana.py"
    spec = importlib.util.spec_from_file_location("grafana_helper", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


try:
    _g = load_grafana_helper()
    _g.load_dotenv()                       # 學員改過 infra/stack/.env 的話，這裡就會吃到新的
    WEEK6_PORTS = {name: _g.port_of(name) for name in _g.DEFAULT_PORTS}
    _port_source = "4_grafana.py (含 infra/stack/.env) "
except Exception as exc:
    # 讀不進來就退回寫死的那一組，至少還有得檢查。
    WEEK6_PORTS = {"replayer06": 18011, "replayer07": 18010,
                   "prometheus": 19090, "grafana": 13000}
    _port_source = f"讀不到 4_grafana.py ({type(exc).__name__}) ，用預設值"
    record("讀得到 4_grafana.py", "注意", _port_source,
           "確認 4_grafana.py 跟這一本放在同一層")

PORT_LABEL = {"replayer06": "Lab 06 重播程式", "replayer07": "Lab 07 重播程式",
              "prometheus": "Prometheus", "grafana": "Grafana"}


def port_status(port):
    """試著綁一次這個 port，回傳 (狀態, 說明) 。

    綁得起來代表沒有別的程式佔著它。綁不起來有兩種情況，要分開講:
      被別的程式佔住            關掉它或換一個 port
      Windows 把這個號碼保留了  沒有程式可以關，只能換一個 port
    """
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe_socket:
        try:
            probe_socket.bind(("0.0.0.0", port))    # Docker 也是綁這個位址，照它的方式試
        except OSError as exc:
            if getattr(exc, "winerror", None) == 10013 or exc.errno == errno.EACCES:
                return "注意", f"port {port} 被 Windows 保留了 (不是別的程式佔住) "
            return "注意", f"port {port} 被別的程式佔住了"
    return "通過", f"port {port} 沒有被佔用"


# 這個 stack 自己已經在跑的時候，它的 port 當然綁不起來，那不是衝突。
OURS = set()
if DAEMON_OK and COMPOSE_OK:
    try:
        OURS = _g.running_services()
    except Exception:
        OURS = set()

for name, port in WEEK6_PORTS.items():
    label = PORT_LABEL.get(name, name)
    if name in OURS:
        status, detail = "通過", f"port {port} 已經是這個 stack 自己在用"
    else:
        status, detail = port_status(port)
    record(f"{label} 的 port {port}", status, detail,
           "關掉佔用的程式，或把 infra/stack/.env.example 複製成 .env，"
           "在裡面換一個沒被佔用的 port")
    rows.append((f"{label} port {port}", status, detail))

show_table(rows, ["項目", "狀態", "實際結果"])
print("port 這一組讀自:", _port_source)

# ---- 4. Docker 沒好的話，把要做的事現在就印出來 ----
#         這一項修起來最久 (要下載 600 MB 到 1 GB) ，所以不等到最後一格才講。
if DOCKER_PATH is None:
    print("\nDocker 還沒裝，這是這一本裡修起來最久的一項，請現在就開始:")
    print("  1. 開瀏覽器到 https://www.docker.com/products/docker-desktop/ 下載 Docker Desktop")
    print("     macOS 要看晶片選版本，這一台是:", platform.machine())
    print("  2. 裝完打開它，等鯨魚圖示不再跑動，再回到這一格按 Shift+Enter 重跑")
    print("  一邊下載一邊往下看後面的檢查，不用停在這裡等。")
elif not DAEMON_OK:
    print("\nDocker 裝好了但沒打開: 打開 Docker Desktop，等圖示不再跑動，回到這一格重跑。")

# ---- 5. Docker 可以用的話，現在就在背景把映像檔抓下來 ----
#         Grafana 那一步要三個映像檔 (Prometheus、Grafana、重播程式的 python 底稿) 。
#         等到下課前才抓，全班會在同一分鐘一起下載; 現在抓的話，學員在跑兩本 lab 的
#         這一個多小時裡它就抓完了。Popen 不等它結束，這一格照樣往下走。
if DAEMON_OK and COMPOSE_OK and PREPULL:
    _compose_file = WEEK6_DIR / "infra" / "stack" / "compose.yaml"
    try:
        subprocess.Popen(["docker", "compose", "-f", str(_compose_file), "pull", "--ignore-buildable"],
                         cwd=str(_compose_file.parent),
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        subprocess.Popen(["docker", "compose", "-f", str(_compose_file), "build"],
                         cwd=str(_compose_file.parent),
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        record("先抓 Grafana 的映像檔", "通過", "已經在背景抓了，不用等它", "")
        print("\n正在背景下載 Grafana 環境要用的映像檔，你可以繼續往下做，不用停在這裡等。")
    except Exception as exc:
        record("先抓 Grafana 的映像檔", "注意", f"{type(exc).__name__}: {exc}",
               "不影響前面兩本 lab，最後一步跑 4_grafana.py 的時候它會自己抓")


In [ ]:
# 🎯 把每一項檢查排成總表，再印出一段可以直接複製貼給講師的純文字。

# ---- 1. 先講結論 ----
from datetime import datetime

passed = [c for c in CHECKS if c["狀態"] == "通過"]
failed = [c for c in CHECKS if c["狀態"] == "失敗"]
warned = [c for c in CHECKS if c["狀態"] == "注意"]

print(f"通過 {len(passed)} 項，失敗 {len(failed)} 項，注意 {len(warned)} 項。")
if failed:
    print("有失敗的項目: 照下面總表「沒過的話怎麼辦」那一欄處理，修完再 Run All Cells 一次。")
else:
    print("沒有失敗的項目，兩本 lab 需要的東西這台機器上都有，可以開始上課。")
if warned:
    print("標成注意的只影響最後一步的 Grafana，前面兩本 lab 的分析照跑。")

# ---- 2. 完整總表 ----
show_table([(c["檢查項目"], c["狀態"], c["實際結果"], c["沒過的話怎麼辦"]) for c in CHECKS],
           ["檢查項目", "狀態", "實際結果", "沒過的話怎麼辦"])

# ---- 3. 給講師的純文字 ----
#         這一段刻意用 print 而不是表格: 表格複製起來會變成沒有對齊的一堆字，
#         純文字可以直接貼進訊息裡。
lines = [
    "===== 環境檢查結果，複製這一整段貼給講師 =====",
    f"檢查時間: {datetime.now():%Y-%m-%d %H:%M}",
    f"作業系統: {platform.platform()} ({platform.machine()}) ",
    f"Python: {platform.python_version()}，直譯器 {sys.executable}",
    f"kernel 環境: {_env_by_var or _env_by_path}" + ("" if IN_COURSE_ENV else f"  (課程環境是 {ENV_NAME}) "),
    f"week6_implementation 資料夾: {WEEK6_DIR}",
    f"結果: 通過 {len(passed)} 項，失敗 {len(failed)} 項，注意 {len(warned)} 項",
]
for check in failed + warned:
    lines.append(f"  [{check['狀態']}] {check['檢查項目']}: {check['實際結果']}")
if not failed and not warned:
    lines.append("  每一項都通過。")
lines.append("=============================================")
print("\n".join(lines))
